In [62]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import faiss
from pathlib import Path
from ollama import embed

In [2]:
### loading the dataset 

ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--qiaojin--PubMedQA. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 19219.74 examples/s]


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

In [7]:
df = pd.DataFrame(ds['train'])  ### creating dataframe from ds

In [8]:
df.head()

,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,17208539,Are the long-term results of the transanal pul...,{'contexts': ['The transanal endorectal pull-t...,Our long-term study showed significantly bette...,no
4,10808977,Can tailored interventions increase mammograph...,{'contexts': ['Telephone counseling and tailor...,The effects of the intervention were most pron...,yes


In [21]:
def get_context_in_list(df:pd.DataFrame) -> list:
    context_list = []
    for _,row in df.iterrows():
        for cntxt in row['context']['contexts']:
            context_list.append(cntxt)

    return context_list


In [ ]:
context_list = get_context_in_list(df) ## saving all context in a list

In [23]:
len(context_list)

3358

In [45]:
### getting embeddings for all context 

context_embeddings = embed(
    model='embeddinggemma',
    input=context_list,
)

In [46]:
context_embeddings = np.array(context_embeddings['embeddings'],dtype= np.float32)

In [48]:
context_embeddings.shape

(3358, 768)

In [49]:
faiss.normalize_L2(context_embeddings)

In [56]:
dim = context_embeddings.shape[1] ### define dim for FAISS
index = faiss.IndexHNSWFlat(dim,32) ## initiating the FAISS

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 50


In [57]:
index.add(context_embeddings)

In [61]:
index.ntotal  ### total number of embeddings saved now

3358

In [63]:
cwd = Path.cwd()
cwd

WindowsPath('c:/Users/ADMIN/MLops/ollama/notebook')

In [ ]:
#### saved index as a file 

index_path = cwd.parent.as_posix() + '/data/processed/medical_rag.index'
faiss.write_index(index,index_path)

In [ ]:
load_index = faiss.read_index(index_path)  ### load index 

In [99]:
query = "what causes cell death?" ## query from user
q_emb = context_embeddings.encode([query])
q_emb = np.array(q_emb,dtype=np.float32)

AttributeError: 'numpy.ndarray' object has no attribute 'encode'

In [87]:
faiss.normalize_L2(q_emb)

In [95]:
scores,indices = index.search(q_emb,10)

In [96]:
scores[0]

array([0.7859874, 1.0399247, 1.1192608, 1.1343892, 1.2313187, 1.2326999,
       1.2493654, 1.2640669, 1.2809361, 1.2841578], dtype=float32)

In [97]:
# CHECK 1 — verify normalization applied correctly
print("chunk norm:", np.linalg.norm(context_embeddings[0]))
# must print: 1.0

# CHECK 2 — verify index built correctly
print("total vectors:", index.ntotal)
# must match number of chunks

# CHECK 3 — verify dimensions
print("dimensions:", context_embeddings.shape)
# must be (num_chunks, 384)

chunk norm: 1.0
total vectors: 3358
dimensions: (3358, 768)


In [ ]:
# ALL GOOD ✅ BUT ONE THING TO NOTE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# RESULTS
# ────────
# chunk norm  : 1.0      ✅ normalized correctly
# total vectors: 3358    ✅ all chunks indexed
# dimensions  : 768      ✅ but different from MiniLM

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 768 DIMENSIONS MEANS
# ─────────────────────
# You are NOT using all-MiniLM-L6-v2 (384 dim)
# You are using a LARGER embedding model

# 768 dim models:
#   all-mpnet-base-v2        ← likely this
#   all-MiniLM-L12-v2
#   embeddinggemma (ollama)

# This is FINE — larger dim = richer embeddings
# just means slightly more memory and compute
# but better semantic understanding ✅

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHICH MODEL ARE YOU USING?
# ───────────────────────────
print(embedder)
# paste output here so we know exact model

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# NOW MOVE TO SEARCH STEP
# ────────────────────────
# CELL — SEARCH
query = "Do mitochondria play a role in cell death?"

# embed query
q_emb = embedder.encode([query])
q_emb = np.array(q_emb).astype("float32")
faiss.normalize_L2(q_emb)

# verify query norm
print("query norm:", np.linalg.norm(q_emb[0]))
# must be 1.0

# verify query dim matches index
print("query dim :", q_emb.shape)
# must be (1, 768) ← same as chunks

# search top 5
scores, indices = index.search(q_emb, 5)
print("scores:", scores[0])
# must be between 0.0 and 1.0
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SyntaxError: invalid character '✅' (U+2705) (3942529000.py, line 1)